# Tuen Mun AQHI prediction — guided analysis
## Step 1 — Understand the input data / 認識輸入資料

Research question: Can information available now predict whether Tuen Mun AQHI will be at least 7 **exactly six hours later**?

We will compare Decision Tree, Random Forest, Logistic Regression, KNN and SVM with a persistence baseline. This first step reads and checks the cleaned input.

**How to run:** In the repository, open this notebook from the repository root. In Google Colab, download `data/processed/tuen_mun_aqhi.csv` from the repository and upload it with the Files sidebar. Run the code cells in order. Python and pandas are required.

Data input comes from the cleaning handoff in repository commit `ba357dd96c231cf2fe2f3141843a25da43d32d05`. The input file's Git blob is `d065826ea12b645554a5fb99296d22001ebf0e11`.


### 1. Read the cleaned CSV / 讀入清理後資料

`pd.read_csv` loads the table into a pandas DataFrame called `data`. A DataFrame is a table with named columns.

- `timestamp`: the end of the observation hour, in Hong Kong time. Source hour 24 becomes midnight on the following day.
- `aqhi`: the current AQHI reading. Project code 11 represents the official 10+ category.
- `is_substituted`: True means EPD used a substitute station reading. These records remain identifiable.

Missing AQHI readings stay empty / NaN. The model target will be the AQHI classification **six hours after** each row.

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("data/processed/tuen_mun_aqhi.csv")
if not data_path.exists():
    data_path = Path("tuen_mun_aqhi.csv")
if not data_path.exists():
    raise FileNotFoundError(
        "Upload tuen_mun_aqhi.csv in Colab, or run from the repository root."
    )

data = pd.read_csv(
    data_path,
    dtype={"aqhi": "float64", "is_substituted": "bool"},
)
data["timestamp"] = pd.to_datetime(
    data["timestamp"], utc=True, errors="raise"
).dt.tz_convert("Asia/Hong_Kong")
data = data.sort_values("timestamp").reset_index(drop=True)

print("First five observations:")
print(data.head().to_string(index=False))


First five observations:
                timestamp  aqhi  is_substituted
2025-04-01 01:00:00+08:00   2.0           False
2025-04-01 02:00:00+08:00   2.0           False
2025-04-01 03:00:00+08:00   2.0           False
2025-04-01 04:00:00+08:00   2.0           False
2025-04-01 05:00:00+08:00   2.0           False


### 2. Check the handoff / 核對交接資料

For a six-hour prediction, the timestamps must be ordered and each row must represent one hour. The checks below verify column names, unique timestamps and hourly continuity.

A missing **measurement** has a timestamp but no AQHI value. A missing **hour** has no row at that time. We count them separately so a later six-hour target can be aligned by time.

In [2]:
assert list(data.columns) == ["timestamp", "aqhi", "is_substituted"]
assert not data.empty
assert data["timestamp"].notna().all()
assert data["timestamp"].is_unique, "Review duplicate timestamps."

expected_hours = pd.date_range(
    start=data["timestamp"].min(),
    end=data["timestamp"].max(),
    freq="h",
)
missing_hours = expected_hours.difference(pd.DatetimeIndex(data["timestamp"]))
assert len(missing_hours) == 0, "Review missing hours before building targets."

available_aqhi = data["aqhi"].dropna()
assert available_aqhi.isin(range(1, 12)).all(), "Review AQHI codes."

quality_summary = pd.Series({
    "Hourly rows": len(data),
    "Available AQHI readings": int(data["aqhi"].notna().sum()),
    "Missing AQHI readings": int(data["aqhi"].isna().sum()),
    "Duplicate timestamps": int(data["timestamp"].duplicated().sum()),
    "Missing hours": len(missing_hours),
    "Substituted readings": int(data["is_substituted"].sum()),
    "10+ category encoded as 11": int(data["aqhi"].eq(11).sum()),
})
print(quality_summary.to_string())
print("First timestamp:", data["timestamp"].min())
print("Last timestamp:", data["timestamp"].max())


Hourly rows                   10944
Available AQHI readings       10926
Missing AQHI readings            18
Duplicate timestamps              0
Missing hours                     0
Substituted readings           1083
10+ category encoded as 11       15
First timestamp: 2025-04-01 01:00:00+08:00
Last timestamp: 2026-07-01 00:00:00+08:00


### 3. Understand the rare category / 認識較少見的類別

The project calls AQHI ≥ 7 the positive class. This cell describes the **current observations**. Future target labels will be built in the next step.

`high_count / available_count` gives the share of available readings at or above 7. Missing values are excluded from this denominator. If the high category is rare, a model can look accurate while missing most high-AQHI periods, so we will compare recall, precision, F1 and error counts.

In [3]:
high_count = int(available_aqhi.ge(7).sum())
other_count = int(available_aqhi.lt(7).sum())
class_counts = pd.Series({
    "AQHI below 7": other_count,
    "AQHI at least 7": high_count,
})
print(class_counts.to_string())
print(f"High-AQHI share among available readings: {high_count / len(available_aqhi):.2%}")
print(f"Available readings below 7: {other_count / len(available_aqhi):.2%}")
print("Missing measurements remain missing:", int(data["aqhi"].isna().sum()))


AQHI below 7       10621
AQHI at least 7      305
High-AQHI share among available readings: 2.79%
Available readings below 7: 97.21%
Missing measurements remain missing: 18


### Checkpoint / 本步完成後要理解的事

1. Each row contains information recorded at one observation time.
2. Missing readings and substituted readings have different meanings and remain distinguishable.
3. Code 11 represents the 10+ category. It preserves ordering, and should be interpreted as a category when explaining results.
4. Counts in this step describe the cleaned observations. Final modeling counts will change when we require past features and a valid six-hour-ahead target.

**Next step:** Build past-only input features and align each row with the AQHI classification exactly six hours later. For example, an input at 09:00 will be paired with the observed category at 15:00. A blank future reading must remain an unknown target.